# YOLOE Object Segmentation
YOLOE (Real-Time Seeing Anything) is a new advancement in zero-shot, promptable YOLO models, designed for **open-vocabulary** detection and segmentation. Unlike previous YOLO models limited to fixed categories, YOLOE uses text, image, or internal vocabulary prompts, enabling real-time detection of any object class. Built upon YOLOv10 and inspired by YOLO-World, YOLOE achieves **state-of-the-art zero-shot performance** with minimal impact on speed and accuracy.
Read more at https://docs.ultralytics.com/models/yoloe/

### Install some dependencies

In [ ]:
pip install --quiet -U supervision opencv-python open-clip-torch timm clip-benchmark datasets torchvision ipywidgets numpy tqdm ultralytics

### Import modules and libraries

In [ ]:
import os
import cv2
import supervision as sv
import matplotlib.pyplot as plt
import numpy as np
from copy import deepcopy
from PIL import Image
from tqdm import tqdm
os.environ['YOLO_VERBOSE'] = 'False'
from ultralytics import YOLOE
from ultralytics.utils.plotting import Annotator, colors

### Specify our source image and display it
The image shape is loaded into height, width and channels so that they can be use later.

In [ ]:
SOURCE_IMAGE_PATH = "images/amsterdam.jpeg"

image = cv2.imread(SOURCE_IMAGE_PATH)
height, width, channels = image.shape
sv.plot_image(image, (20,20))

### Set up the inferencing 

A pre-trained model is loaded and and a set of classes to detect is configured.

A coule of options are specified when inferencing the model

- `conf=0.11           ` This adjusts the confidence level of the model and can be adjusted to change ddetection accuracay\
- `imgsz=(height,width)` Specifies that the photo should be resized to it's original size.  Without this the image would have been reduced to 640px wide. If the images is not a multiple of 32, then the image will be automatically resized and cropped to meet this requirement.
- `verbose=False       ` Prevents the display of output from the server inferencing 

Required models, including `mobileclip_blt.pt` used for text processing, will be downloaded with this step if they are not present on the system.

In [ ]:
# Load a pretrained YOLOE model that provides image segmentation.
model = YOLOE("yoloe-11s-seg.pt")

# Define custom classes
names = ["flowers", "bag", "bicycle sign"]
model.set_classes(names, model.get_text_pe(names))

# Execute prediction for specified categories on an image
results = model(SOURCE_IMAGE_PATH, conf=0.11, imgsz=(height,width), verbose=False)

### Display the results
To change the font size, the image must be converted to PIL format (`pil=True`).  We are able to convert is back to BGR format using cv2 and numpy.

Options that are specified are:

- `conf=False  ` This prevents the confidencce score from being shown.  When adjusting the model confidence it is handy to turn this option to "True"\
- `line_width=1` This makes the bounding box only 1 pixel wide and enhances the visibility of the segmented objects.\
- `font_size=20` Specifies the font size for the label.


In [ ]:
for result in results:
    
    # Changing the font size changes the image to a PIL array which we need to change
    # back to a numpy array
    img_pil = results[0].plot(conf=False, line_width=1, font_size=20, pil=True)
    rgb_array = np.array(img_pil)
    
    # Convert RGB to BGR using OpenCV and show the image
    bgr_array = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)
    image = Image.fromarray(bgr_array)
    sv.plot_image(image, (20,20))

# YOLOE Segmentation in Videos

### Load the model and determine set the aspect ratio
The aspect ration of the model must be set as a multiple of 32 for both the height and the width to optimize performance.  The `desired_w` and `desired_h` variables set the desired dimentions of each frame that will be processed.  THe higher the resolution, the better the inferencing will be.  Upscaling a video that alrady has poor resolution may not have the desired effect so it is recommended to not adjust the desired aspect ration higher than the videos native resolution.

In [ ]:
# Load a pretrained YOLOE model that provides image segmentation.
model = YOLOE("yoloe-11s-seg.pt")

# set the source and target videos variables
SOURCE_VIDEO_PATH = f"videos/soccer-game.mp4"
TARGET_VIDEO_PATH = f"soccer_output.mp4"

# get information about the video (height, width) and calulate so that they are miltiples of 32.  This will be needed later to perform inference on the video.
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)  

# Set the desired height and width and get the frame size
desired_w = 1920
desired_h = 1088

width, height = video_info.resolution_wh
aspect= width/height

# Calculate the correct height and width to use in inferencing
if height > desired_h:
    height = desired_h
else:
    multiplier = round(height/32)
    height = 32 * multiplier

width = height * aspect

if width > desired_w:
    width = desired_w
else:
    multiplier = round(width/32)
    width = 32 * multiplier

### Test if the settings produce good results
`objects` is an array of what will be looked for in the video.\
Other variables that are used are similar to object detection in an image.

Inferencing Options:
- `conf=0.40    `  The confidence that the model should operate with
- `imgsz=(h,w)  `  The size of the video that was previously determined
- `verbose=False`  Remove the output from the server inference
- `stream=True  `  The input will be a stream of data\

Output options:
- `line_width=2 `  The width of the bounding box
- `font_size=60 `  The size of the font that is used in the label
- `pil=True     `  Needed to adjust the font size

If the processed frame does not look good, adjust the variables or ratio settings in the previous section to improve the inferencing results.

In [ ]:
generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
frame = next(generator)

# rawframe = cv2.resize(frame, (width, height))
rawframe = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
frame = Image.fromarray(rawframe) 

# Place the objects that we want to detect in to the array
objects = ["soccer ball"]
model.set_classes(objects, model.get_text_pe(objects))

results = model(frame, conf=0.05, imgsz=(height,width), verbose=False, stream=True)
for result in results:
    
    # Changing the font size changes the image to a PIL array which we need to change
    # back to a numpy array
    img_pil = result[0].plot(conf=False, line_width=1, font_size=20, pil=True)
    rgb_array = np.array(img_pil)
    
    # Convert RGB to BGR using OpenCV and show the image
    bgr_array = cv2.cvtColor(rgb_array, cv2.COLOR_RGB2BGR)
    image = Image.fromarray(bgr_array)
    sv.plot_image(image, (20,20))

### Inference the video
At this point, the entire video can be inferenced to locate the objects that we specified earlier.\
Other variables that are used are similar to object detection in an image.\

Inferencing Options:
- `conf=0.30    ` The confidence that the model should operate with
- `imgsz=(h,w)  ` The size of the video that was previously determined
- `verbose=False` Remove the output from the server inference
- `stream=True  ` The input will be a stream of data\

Out Options:
- `labels=False ` Do not display the label of the confidence level
- `line_width=2 ` The width of the bounding box
- `pil=True     ` Needed to adjust the font size

In [ ]:
frame_generator = sv.get_video_frames_generator(SOURCE_VIDEO_PATH)
video_info = sv.VideoInfo.from_video_path(SOURCE_VIDEO_PATH)

# Place the objects that we want to detect in to the array
objects = ["soccer ball"]
model.set_classes(objects, model.get_text_pe(objects))

# Read the video one frame at a time and do inference
with sv.VideoSink(target_path=TARGET_VIDEO_PATH, video_info=video_info) as sink:
    for frame in tqdm(frame_generator, total=video_info.total_frames):

        #Convert the frame to the right format for inference
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
        frame = Image.fromarray(frame) 

        # Do model infreence.  Note that streaming is set to true.
        results = model(frame, conf=0.30, imgsz=(height,width), verbose=False, stream=True)

        for result in results:

            # Adjust the font size, labels, line width and output as a PIL Image
            img_pil = result.plot(labels=False, boxes=False, pil=True)
        
            # Convert the PIL to RGB
            rgb_array = np.array(img_pil)
        
            # Write the RBG array to the output sink 
            sink.write_frame(rgb_array)